In [1]:
import os
import sys


In [2]:
pwd


'/home/pavani/datascience_e_2_e/networksecurity1/notebook'

In [3]:
import os


In [4]:
os.chdir("../")
%pwd

'/home/pavani/datascience_e_2_e/networksecurity1'

In [13]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataIngestionConfig:
    root_dir: Path
    feature_store_file_path: Path
    training_file_path: Path
    testing_file_path: Path
    train_test_split_ratio: float
    collection_name: str
    database_name: str

In [14]:
from src.networksecurity.constants import *
from src.networksecurity.utils.common import read_yaml, create_directories
from src.networksecurity.entity.config_entity import DataIngestionConfig


In [15]:
import os
from datetime import datetime
from src.networksecurity.constants import *
from src.networksecurity.utils.common import read_yaml, create_directories
from src.networksecurity.entity.config_entity import DataIngestionConfig

class TrainingPipelineConfig:
    def __init__(self):
        # Read the artifacts_root from config.yaml
        config = read_yaml(CONFIG_FILE_PATH)
        self.timestamp = datetime.now().strftime("%m_%d_%Y_%H_%M_%S")
        self.artifact_root = os.path.join(config.artifacts_root, self.timestamp)

class Configurationmanager:
    def __init__(self, 
                 config_filepath = CONFIG_FILE_PATH,
                 params_filepath = PARAMS_FILE_PATH,
                 schema_filepath = SCHEMA_FILE_PATH):
        
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)
        
        # Initialize the versioning class
        self.pipeline_config = TrainingPipelineConfig()
        
        # Create the top-level timestamped directory
        create_directories([self.pipeline_config.artifact_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion
        params = self.params.data_ingestion
        
        # Assemble path: artifacts/timestamp/data_ingestion
        data_ingestion_dir = os.path.join(self.pipeline_config.artifact_root, config.root_dir)
        create_directories([data_ingestion_dir])

        return DataIngestionConfig(
            root_dir = data_ingestion_dir,
            feature_store_file_path = os.path.join(data_ingestion_dir, config.feature_store_file_name),
            training_file_path = os.path.join(data_ingestion_dir, config.train_file_name),
            testing_file_path = os.path.join(data_ingestion_dir, config.test_file_name),
            train_test_split_ratio = params.train_test_split_ratio,
            collection_name = config.collection_name,
            database_name = config.database_name
        )
        




In [16]:
import os
import sys
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import pymongo
from typing import List
from src.networksecurity.exception.exception import NetworkSecurityException
from src.networksecurity.logging import logger
from src.networksecurity.entity.config_entity import DataIngestionConfig
from src.networksecurity.entity.artifact_entity import DataIngestionArtifact
from src.networksecurity.utils.common import *
from dotenv import load_dotenv
load_dotenv()

MONGO_DB_URL = os.getenv("MONGO_DB_URL")

class DataIngestion:
    def __init__(self, data_ingestion_config: DataIngestionConfig):
        """ 
        Initializes the DataIngestion component.
        Ensure Your MONGO_DB_URL is set in your environment variables
                                                                    """
        try: 
            self.data_ingestion_config = data_ingestion_config
            self.mongo_db_url = os.getenv("MONGO_DB_URL")
        except Exception as e:
            raise NetworkSecurityException(e, sys)
        
    def export_collection_as_dataframe(self):
        try:
            database_name = self.data_ingestion_config.database_name
            collection_name = self.data_ingestion_config.collection_name
            collection = self.mongo_clinet[database_name][collection_name]

            df = pd.DataFrame(list(collection.find()))
            if "_id" in df.columns.to_list():
                df = df.drop(columns=["_id"], axis = 1)

            df.replace({"na": np.nan}, inplace = True)
            return df
        
        except Exception as e:
            raise NetworkSecurityException(e, sys)
        
    def export_data_into_feature_store(self, dataframe: pd.DataFrame) -> pd.DataFrame:
        try:
            feature_store_file_path = self.data_ingestion_config.feature_store_file_path


            create_directories([os.path.dirname(feature_store_file_path)])
            save_data(dataframe, self.data_ingestion_config.feature_store_file_path)
            return dataframe
        except Exception as e:
            raise NetworkSecurityException(e, sys)
        
    def split_data_as_train_test(self, dataframe:pd.DataFrame):
        try:
           logger.info("splitting data into train_test_split_ratio")
           train_set, test_set = train_test_split(dataframe ,test_size = self.data_ingestion_config.train_test_split_ratio)

           #saving training data
           create_directories([os.path.dirname(self.data_ingestion_config.training_file_path)])
           save_data(train_set, self.data_ingestion_config.training_file_path)

           #saving testing data
           create_directories([os.path.dirname(self.data_ingestion_config.testing_file_path)])
           save_data(test_set, self.data_ingestion_config.testing_file_path)

           logger.info("Train/Test split complet and saved")

        except Exception as e:
            raise NetworkSecurityException(e, sys)
        
    def initiate_data_ingestion(self) -> DataIngestionArtifact:
        try:
            dataframe = self.export_collection_as_dataframe()
            dataframe = self.export_data_into_feature_store(dataframe)
            self.split_data_as_train_test(dataframe)

            #Returning the artifact object
            data_ingestion_artifact = DataIngestionArtifact(
                trained_file_path= self.data_ingestion_config.training_file_path,
                test_file_path = self.data_ingestion_config.testing_file_path)
            return data_ingestion_artifact
        
        except Exception as e:
            raise NetworkSecurityException(e, sys)
             

        
        


        
    
        



In [17]:
from src.networksecurity.config.configuration import Configurationmanager
from src.networksecurity.components.data_ingestion import DataIngestion
from src.networksecurity.logging import logger